In [1]:
from pathlib import Path
import os
os.chdir(Path("~/Documents/vggt2/digitizing_reality/TRELLIS").expanduser())

In [ ]:
import os
# os.environ['ATTN_BACKEND'] = 'xformers'   # Can be 'flash-attn' or 'xformers', default is 'flash-attn'
os.environ['SPCONV_ALGO'] = 'native'        # Can be 'native' or 'auto', default is 'auto'.
                                            # 'auto' is faster but will do benchmarking at the beginning.
                                            # Recommended to set to 'native' if run only once.

import imageio
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

# Load a pipeline from a model folder or a Hugging Face model hub.
pipeline = TrellisImageTo3DPipeline.from_pretrained("microsoft/TRELLIS-image-large")
pipeline.cuda()

# Load an image
image = Image.open("../data/frame_B0.jpg")

# Run the pipeline
outputs = pipeline.run(
    image,
    seed=1,
    # Optional parameters
    # sparse_structure_sampler_params={
    #     "steps": 12,
    #     "cfg_strength": 7.5,
    # },
    # slat_sampler_params={
    #     "steps": 12,
    #     "cfg_strength": 3,
    # },
)
# outputs is a dictionary containing generated 3D assets in different formats:
# - outputs['gaussian']: a list of 3D Gaussians
# - outputs['radiance_field']: a list of radiance fields
# - outputs['mesh']: a list of meshes

# Render the outputs
video = render_utils.render_video(outputs['gaussian'][0])['color']
imageio.mimsave("../data/TRELLIS_gs.gif", video, fps=30, loop=0)
# video = render_utils.render_video(outputs['radiance_field'][0])['color']
# imageio.mimsave("../data/sample_rf.mp4", video, fps=30)
video = render_utils.render_video(outputs['mesh'][0])['normal']
imageio.mimsave("../data/TRELLIS_mesh.gif", video, fps=30, loop=0)

# GLB files can be extracted from the outputs
glb = postprocessing_utils.to_glb(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    # Optional parameters
    simplify=0.95,          # Ratio of triangles to remove in the simplification process
    texture_size=1024,      # Size of the texture used for the GLB
)
glb.export("../data/TRELLIS.glb")

# Save Gaussians as PLY files
outputs['gaussian'][0].save_ply("../data/TRELLIS.ply")

[SPARSE] Backend: spconv, Attention: flash_attn
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[SPARSE][CONV] spconv algo: native


ckpts/ss_flow_img_dit_L_16l8_fp16.safete(…):   0%|          | 0.00/1.13G [00:00<?, ?B/s]

[ATTENTION] Using backend: flash_attn


slat_dec_gs_swin8_B_64l8gs32_fp16.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.(…):   0%|          | 0.00/171M [00:00<?, ?B/s]

...

Decimating Mesh: 100%|██████████[00:01<00:00]


After decimate: 13815 vertices, 27658 faces


Rasterizing: 100%|██████████| 1000/1000 [00:00<00:00, 1801.37it/s]


Found 338 invisible faces
Dual graph: 41487 edges
Mincut solved, start checking the cut
Removed 350 faces by mincut

Loading ..98%

100% done 
After remove invisible faces: 13649 vertices, 27330 faces


Rendering: 100it [00:00, 266.31it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:06<00:00, 376.42it/s, loss=0.0495]


In [ ]:

video = render_utils.render_video(outputs['mesh'][0])['color']
imageio.mimsave("../data/TRELLIS_mesh_color.gif", video, fps=30, loop=0)

Rendering: 300it [00:02, 127.04it/s]


KeyError: 'color'

Rendering: 300it [00:02, 127.09it/s]


dict_keys(['normal'])

In [3]:
# !pip install kaolin

import torch, sys

torch_ver = torch.__version__.split("+")[0]          # e.g. "2.1.0"
cuda_ver = torch.version.cuda.replace(".", "")       # e.g. "121"
print(torch_ver, cuda_ver)

2.4.0 121


In [13]:
!pip install kaolin==0.18.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in links: https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 5.4 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 27.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 MB 31.9 MB/s  0:00:04m0:00:0100:01
  Attempting uninstall: jupyter_client━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [usd-core]]
    Found existing installation: jupyter_client 8.8.0━━━━━━━━━  2/14 [usd-core]
    Uninstalling jupyter_client-8.8.0:━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [usd-core]
      Successfully uninstalled jupyter_client-8.8.0━━━━━━━━━━━  2/14 [usd-core]
  Attempting uninstall: kaolin━━━━━━━━━╺━━━━━━━━ 11/14 [ipyevents]
    Found existing installation: kaolin 0.1m╺━━━━━━━━ 11/14 [ipyevents]
    Uninstalling kaolin-0.1:━━━━━━━╺━━━━━━━━ 11/14 [ipyevents]
      Successfully uninstalled kaolin-0.190m